# General model update

Use this notebook after `01_training_collection_preprocess_train_validate.ipynb` has trained the normal run-specific model. Notebook 01 remains the regular per-run training workflow. This notebook only initializes or updates the cumulative `general_model.pt`.

For the most stable superdecoder, update the general model with replay data from previous runs as well as the newest run. Updating only on the newest run is supported, but it can make the model forget older runs.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'training.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from modeling import class_names_from_checkpoint, load_checkpoint, window_config_from_checkpoint
from training import (
    TrainingConfig,
    WindowConfig,
    initialize_general_model,
    labeled_training_paths_for_runs,
    update_general_model,
)

print('Pipeline root:', ROOT)


## Settings

`RUN_ID` is the newest run you want to add. If `general_model.pt` does not exist yet, this notebook can initialize it by copying the run-specific checkpoint from that run. If `general_model.pt` already exists, it continues training from that checkpoint.

Set `USE_REPLAY_RUNS = True` and list multiple `TRAIN_RUN_IDS` to reduce forgetting and build a more general decoder.


In [ ]:
RUN_ID = 'run_001'
RUNS_ROOT = ROOT / 'runs'
RUN_DIR = RUNS_ROOT / RUN_ID
RUN_MODEL_CHECKPOINT = RUN_DIR / 'lstm_checkpoint.pt'

GENERAL_MODEL_DIR = ROOT / 'general_models'
GENERAL_MODEL_PATH = GENERAL_MODEL_DIR / 'general_model.pt'
UPDATE_DIR = GENERAL_MODEL_DIR / 'updates' / RUN_ID
SNAPSHOT_PATH = GENERAL_MODEL_DIR / f'general_model_after_{RUN_ID}.pt'
LOG_CSV = GENERAL_MODEL_DIR / 'general_model_training_log.csv'

INITIALIZE_IF_MISSING = True
CONTINUE_AFTER_INITIALIZE = False  # Usually False for run_001 so the run model is not trained twice.

USE_REPLAY_RUNS = False
TRAIN_RUN_IDS = (RUN_ID,)  # Example replay setting: ('run_001', 'run_002', 'run_003')
if not USE_REPLAY_RUNS:
    TRAIN_RUN_IDS = (RUN_ID,)

WIN = WindowConfig(
    feature_mode='filtered_signal',
    window_sec=2.0,
    stride_sec=0.05,
    label_mode='endpoint',
)

TRAIN_UPDATE = TrainingConfig(
    train_fraction=1.0,
    hidden_size=64,      # ignored when continuing from general_model.pt; checkpoint architecture is reused.
    num_layers=2,        # ignored when continuing from general_model.pt; checkpoint architecture is reused.
    dropout=0.25,        # ignored when continuing from general_model.pt; checkpoint architecture is reused.
    batch_size=64,
    epochs=10,
    lr=3e-4,
    seed=888,
)

GENERAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
UPDATE_DIR.mkdir(parents=True, exist_ok=True)
print('Newest run:', RUN_ID)
print('Training data run IDs:', TRAIN_RUN_IDS)
print('Run checkpoint:', RUN_MODEL_CHECKPOINT)
print('General model:', GENERAL_MODEL_PATH)
print('Update output:', UPDATE_DIR)


## Resolve update inputs

These are the labeled training files that will be used for the general-model update. With replay enabled, this list should include older runs plus the newest run.


In [ ]:
labeled_paths = labeled_training_paths_for_runs(RUNS_ROOT, TRAIN_RUN_IDS)
print(f'Found {len(labeled_paths)} labeled training file(s):')
for path in labeled_paths:
    print(' ', path)

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Run folder not found: {RUN_DIR}')
if not GENERAL_MODEL_PATH.exists() and not RUN_MODEL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f'Cannot initialize general model because run checkpoint is missing: {RUN_MODEL_CHECKPOINT}'
    )


## Initialize or update general model

If `general_model.pt` is missing, it is initialized by copying the current run checkpoint. If it already exists, it is continued using the labeled training files selected above. The updated checkpoint is saved back to `general_model.pt`, and a run-specific snapshot is saved as `general_model_after_<RUN_ID>.pt`.


In [ ]:
init_result = None
update_result = None

if not GENERAL_MODEL_PATH.exists():
    if not INITIALIZE_IF_MISSING:
        raise FileNotFoundError(f'General model not found: {GENERAL_MODEL_PATH}')
    init_result = initialize_general_model(
        source_checkpoint_path=RUN_MODEL_CHECKPOINT,
        general_model_path=GENERAL_MODEL_PATH,
        snapshot_path=SNAPSHOT_PATH if not CONTINUE_AFTER_INITIALIZE else None,
        update_name=RUN_ID,
        log_csv=LOG_CSV,
    )
    print('Initialized general model from run checkpoint.')
    display(pd.DataFrame([{k: str(v) for k, v in init_result.items()}]))

if GENERAL_MODEL_PATH.exists() and (CONTINUE_AFTER_INITIALIZE or init_result is None):
    update_result = update_general_model(
        labeled_npz_paths=labeled_paths,
        general_model_path=GENERAL_MODEL_PATH,
        output_dir=UPDATE_DIR,
        training_config=TRAIN_UPDATE,
        window_config=WIN,
        snapshot_path=SNAPSHOT_PATH,
        update_name=RUN_ID,
        log_csv=LOG_CSV,
    )
    print('Updated general model with selected labeled data.')
    display(update_result['validation_summary'])
else:
    print('General model was initialized only. Set CONTINUE_AFTER_INITIALIZE=True to also fine-tune it on this run.')


## Inspect general model

This confirms the active `general_model.pt` settings and shows the update log. If this notebook performed continued training, it also plots the update training history.


In [ ]:
model, checkpoint, device = load_checkpoint(GENERAL_MODEL_PATH)
win_cfg = window_config_from_checkpoint(checkpoint)
inspect_rows = [{
    'general_model_path': str(GENERAL_MODEL_PATH),
    'device': str(device),
    'class_names': class_names_from_checkpoint(checkpoint),
    'feature_mode': win_cfg.feature_mode,
    'window_sec': win_cfg.window_sec,
    'stride_sec': win_cfg.stride_sec,
    'label_mode': win_cfg.label_mode,
    'source_files': checkpoint.get('source_files', ()),
    'continued_from_checkpoint': checkpoint.get('continued_from_checkpoint', ''),
}]
display(pd.DataFrame(inspect_rows))

if LOG_CSV.exists():
    display(pd.read_csv(LOG_CSV).tail(10))

if update_result is not None:
    history = update_result['history']
    display(history.tail())
    ax = history.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(10, 4), marker='o')
    ax.set_title(f'General model update loss: {RUN_ID}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    ax = history.plot(x='epoch', y=['train_acc', 'val_acc'], figsize=(10, 4), marker='o')
    ax.set_title(f'General model update accuracy: {RUN_ID}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
